# Lab 12 — Safety and Governance

## Objective

Validate requests, restrict actions by role, require scoped approval and log decisions.

## Introduction and concept

handle rejects malformed destinations and restricted actions, and ties the approval fixture to both the action and record.

## Real-world use case

A teacher reviews a request to release a specific student result.

## Technology stack and dependencies

Python dataclasses and local audit log. Use Python 3.11 or newer and a Jupyter kernel from the environment where you installed the [assignment requirements](../requirements.txt). Start the notebook from its lab directory.

## Architecture and workflow

Validate → Check role → Check approval → Record decision


## Step 1 — Implement the components

handle rejects malformed destinations and restricted actions, and ties the approval fixture to both the action and record.

In [1]:
from dataclasses import dataclass

@dataclass(frozen=True)
class Approval:
    action: str
    record_id: str
    reviewer: str

PERMISSIONS = {"student": {"view_status"}, "teacher": {"view_status", "release_result"}}

def handle(actor, action, record_id, approval, audit):
    valid = (
        isinstance(action, str) and isinstance(record_id, str)
        and record_id.startswith("demo-") and len(record_id) > 5
    )
    if not valid:
        outcome = "invalid_request"
    elif action not in PERMISSIONS.get(actor, set()):
        outcome = "denied"
    elif action == "release_result" and approval != Approval(action, record_id, "teacher-reviewer"):
        outcome = "awaiting_review"
    else:
        outcome = "allowed_in_demo"
    audit.append({"actor": actor, "outcome": outcome})
    return outcome

## Step 2 — Run sample inputs and inspect actual output

Try valid student access, forbidden release and deletion, an invalid record ID, a mismatched approval and an approved release.

In [2]:
def main():
    audit = []
    approval = Approval("release_result", "demo-42", "teacher-reviewer")
    cases = [
        ("student", "view_status", "demo-42", None, "allowed_in_demo"),
        ("student", "release_result", "demo-42", approval, "denied"),
        ("teacher", "delete_recording", "demo-42", None, "denied"),
        ("teacher", "release_result", "../invalid", None, "invalid_request"),
        ("teacher", "release_result", "demo-43", approval, "awaiting_review"),
        ("teacher", "release_result", "demo-42", approval, "allowed_in_demo"),
    ]
    for actor, action, record_id, review, expected in cases:
        outcome = handle(actor, action, record_id, review, audit)
        assert outcome == expected
        print(actor, action, outcome)
    assert len(audit) == len(cases)
    print("PASS: authorization, approval scope and audit trail")

main()

student view_status allowed_in_demo
student release_result denied
teacher delete_recording denied
teacher release_result invalid_request
teacher release_result awaiting_review
teacher release_result allowed_in_demo
PASS: authorization, approval scope and audit trail


## Testing and validation

Try valid student access, forbidden release and deletion, an invalid record ID, a mismatched approval and an approved release. Assertions in the preceding cell check the demonstrated behavior and stop execution on a failed expectation.

## Limitations

Roles and approvals are trusted coursework fixtures. Production authorization requires authenticated identity and durable secure audits.

## How this relates to ExamPilot

Implemented safeguards and partial governance analogue. ExamPilot applies role-aware authorization and answer-key isolation. See backend/app/agents/exam_session_agent/exam_session_agent.py.

## Conclusion

Validate requests, restrict actions by role, require scoped approval and log decisions. Use the trace and assertions to explain which decisions the workflow makes.